In [ ]:
import gymnasium as gym
import pde_control_gym
import numpy as np
import math
import matplotlib.pyplot as plt
import stable_baselines3
import time
from utils import set_size
from utils import linestyle_tuple
from utils import load_csv
from stable_baselines3 import PPO
from stable_baselines3 import SAC
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import CheckpointCallback

In [ ]:
# Print Versioning
print("Gym version", gym.__version__)
print("Numpy version", np.__version__)
print("Stable Baselines3 version", stable_baselines3.__version__)

In [ ]:
# NO NOISE
def noiseFunc(state):
    return state

# Chebyshev Polynomial Beta Functions
def solveBetaFunction(x, gamma):
    beta = np.zeros(len(x), dtype=np.float32)
    for idx, val in enumerate(x):
        beta[idx] = 5*math.cos(gamma*math.acos(val))
    return beta

# Kernel function solver for backstepping
def solveKernelFunction(theta):
    kappa = np.zeros(len(theta))
    for i in range(0, len(theta)):
        kernelIntegral = 0
        for j in range(0, i):
            kernelIntegral += (kappa[i-j]*theta[j])*dx
        kappa[i] = kernelIntegral  - theta[i]
    return np.flip(kappa)

# Control convolution solver
def solveControl(kernel, u):
    res = 0
    for i in range(len(u)):
        res += kernel[i]*u[i]
    return res*1e-2

# Set initial condition function here
def getInitialCondition(nx):
    return np.ones(nx)*np.random.uniform(1, 10)

# Returns beta functions passed into PDE environment. Currently gamma is always
# set to 7.35, but this can be modified for further problesms
def getBetaFunction(nx):
    return solveBetaFunction(np.linspace(0, 1, nx), 7.35)


In [ ]:
# Timestep and spatial step for PDE Solver
T = 240
dt = 1
dx = 10
X = 500

In [ ]:
# Backstepping does not need to normalize actions to be between -1 and 1, so normalize is set to False. Otherwise, 
# parameters are same as RL algorithms
from pde_control_gym.src import TunedReward1D
reward_class =  TunedReward1D(int(round(T/dt)), -1e3, 3e2)

hyperbolicParameters = {
        "T": T, 
        "dt": dt, 
        "X": X,
        "dx": dx, 
        "reward_class": reward_class,
        "normalize":None, 
        # "sensing_loc": "full", 
        # "control_type": "Dirchilet", 
        # "sensing_type": None,
        # "sensing_noise_func": lambda state: state,
        # "limit_pde_state_size": True,
        # "max_state_value": 1e10,
        # "max_control_value": 20,
        # # "reset_init_condition_func": getInitialCondition,
        # "reset_recirculation_func": getBetaFunction,
        # "control_sample_rate": 0.1,
        "v_steady" : 36,
        "ro_steady" : 0.12,
        "v_max" : 40,
        "ro_max" : 0.16,
        "v_desired" : 10,
        "ro_desired" : 0.12,
        "tau" : 30
}

hyperbolicParametersBackstepping = hyperbolicParameters.copy()
hyperbolicParametersBackstepping["normalize"] = False

hyperbolicParametersRL = hyperbolicParameters.copy()
hyperbolicParametersRL["normalize"] = True